# Export سایکل‌های v2 (آشکارساز fused) روی Colab — تغییر cycle-detection-v3-multindex

طبق سیاست compute-placement (AGENTS §6، دستور مالک ۲۰۲۶-۱۱-۲۰): export طولانی روی Colab اجرا می‌شود؛ گیت محلی (تست + smoke ۸ مزرعه + بازبینی خروجی) قبلاً پاس شده است.

**قبل از اجرا:**
1. کامیت محلی `6c9d92f` باید push شده باشد (سلول ۲ خودش چک می‌کند)
2. بستهٔ `phenov2_statszarr_2026-11-20.zip` در `MyDrive/thesis_colab/` باشد

**خروجی:** `cycles_490_v2.parquet` + `growth_cycles_490_v2.npz` + provenance — به‌صورت `phenov2_cyclev2_2026-11-20.zip` به Drive برمی‌گردد. زمان تخمینی: ۵ تا ۱۰ دقیقه (CPU؛ GPU لازم نیست).

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr==3.3.0"], check=True)
r = subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Mahdikalantari555/thesis_implementation.git",
                    "/content/thesis_implementation"])
assert r.returncode == 0, "clone ناموفق — ریپو باید public و کامیت‌ها push شده باشند"
script = open("/content/thesis_implementation/scripts/segment_and_export_cycles_490.py").read()
assert "--detector" in script and "fused" in script, (
    "کد clone‌شده حالت fused ندارد — کامیت 6c9d92f را push کنید و سلول را دوباره اجرا کنید")
import subprocess as sp
print("HEAD:", sp.run(["git", "-C", "/content/thesis_implementation", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

In [ ]:
from google.colab import drive
import pathlib, shutil, zipfile

drive.mount("/content/drive")
ZIP = pathlib.Path("/content/drive/MyDrive/thesis_colab/phenov2_statszarr_2026-11-20.zip")
assert ZIP.is_file(), f"بسته پیدا نشد — آن را در {ZIP} بگذارید"
shutil.copy(ZIP, "/content/stats_bundle.zip")
with zipfile.ZipFile("/content/stats_bundle.zip") as z:
    z.extractall("/content/thesis_implementation/")
    print("extracted:", len(z.namelist()), "entries")

In [ ]:
import os, subprocess, sys, time

os.chdir("/content/thesis_implementation")
env = dict(os.environ, PYTHONPATH="/content/thesis_implementation/src")
t0 = time.time()
r = subprocess.run(
    [sys.executable, "-X", "utf8", "-u",
     "scripts/segment_and_export_cycles_490.py",
     "--zarr", "Data/Processed/satellite/stats.zarr",
     "--detector", "fused",
     "--cycles-out", "data/processed/cycles_490_v2.parquet",
     "--npz-out", "data/interim/growth_cycles_490_v2.npz"],
    env=env,
)
print(f"exit={r.returncode}  wall={time.time()-t0:.0f}s")
assert r.returncode == 0, "export ناموفق — خروجی بالا را کامل بفرستید"

In [ ]:
import json, os, shutil, zipfile

os.chdir("/content/thesis_implementation")
prov = json.load(open("data/processed/cycles_490_v2.provenance.json"))
print("v1<->v2 comparison:", json.dumps(prov["comparison"], indent=2))
art = ["data/processed/cycles_490_v2.parquet",
       "data/processed/cycles_490_v2.provenance.json",
       "data/interim/growth_cycles_490_v2.npz"]
with zipfile.ZipFile("/content/phenov2_cyclev2_2026-11-20.zip", "w") as z:
    for a in art:
        z.write(a, a)
        print("bundled:", a, f"({os.path.getsize(a)/1e3:.0f} KB)")
dest = "/content/drive/MyDrive/thesis_colab/phenov2_cyclev2_2026-11-20.zip"
shutil.copy("/content/phenov2_cyclev2_2026-11-20.zip", dest)
print("به Drive برگشت:", dest)

## مسیر برگشت (سیاست §6.2)

1. `phenov2_cyclev2_2026-11-20.zip` را از Drive دانلود کنید و به ایجنت محلی بدهید.
2. ایجنت محلی: گیت‌های D6 روی جدول v2 → `splits_locked_v2.json` + `holdout_manifest_v2.json` → سپس دستور استخراج محلی (۳–۴ ساعت، چون آرشیو ۴۴ گیگابایتی صحنه‌ها فقط روی سیستم شماست).
3. هیچ artifact نسخهٔ ۱ نوشته یا بازنویسی نمی‌شود؛ هر دو نسخه کنار هم می‌مانند (تحویل‌دادنی مقایسهٔ v1↔v2).